## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [ ]:
import numpy as np
import pandas as pd
import tqdm.notebook as tqdm
import scipy.sparse as sps
from scipy import stats
import gc

from Challenge.paths import load_holdout_split
from Challenge.utils import load_models

DATAFRAMES_PATH = os.path.join(WORKING_DIR, "xg_boost_dataframes")
os.makedirs(DATAFRAMES_PATH, exist_ok=True)

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


## **Load Data**

In [ ]:
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

URM_train_complete, URM_test = load_holdout_split()
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_complete, train_percentage = 0.8)

## **Recommeder List**

In [ ]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAERecommender import MultVAERecommender

models_mapping = {
    'TopPop': TopPop,
    'ItemKNNCF_cosine': ItemKNNCFRecommender,
    'ItemKNNCF_jaccard': ItemKNNCFRecommender,
    'ItemKNNCF_asymmetric': ItemKNNCFRecommender,
    'ItemKNNCF_tversky': ItemKNNCFRecommender,
    'ItemKNNCF_dice': ItemKNNCFRecommender,
    'UserKNNCF_cosine': UserKNNCFRecommender,
    'UserKNNCF_jaccard': UserKNNCFRecommender,
    'UserKNNCF_asymmetric': UserKNNCFRecommender,
    'UserKNNCF_tversky': UserKNNCFRecommender,
    'UserKNNCF_dice': UserKNNCFRecommender,
    'MultiThreadSLIM_SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'EASE_R': EASE_R_Recommender,
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': IALSRecommender,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    'MatrixFactorization_SVDpp': MatrixFactorization_SVDpp_Cython,
    
    'SLIM_BPR': SLIM_BPR_Cython,
    'NMF': NMFRecommender,
    'MultVAE': MultVAERecommender,
}

candidate_mapping = {
    'MultiThreadSLIM_SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'ItemKNNCF_tversky': ItemKNNCFRecommender,
    'RP3betaRecommender': RP3betaRecommender,
    'TopPop': TopPop
}

candidate_cutoff = {
    'MultiThreadSLIM_SLIMElasticNet': 100,
    'ItemKNNCF_tversky': 100,
    'RP3betaRecommender': 100,
    'TopPop': 100
}

Tensorflow is not available


## **Functions**

In [ ]:
def generate_candidates(URM_train):
    candidate_models = load_models(URM_train, candidate_mapping, model_folder="xg_boost_train_candidates")

    n_users, n_items = URM_train.shape

    training_dataframe = pd.DataFrame(index=range(0, n_users), columns = ["ItemID"])
    training_dataframe.index.name='UserID'

    for model_name, recommender in candidate_models:
        cutoff = candidate_cutoff[model_name]
        
        for user_id in tqdm(range(n_users)):
            recommendations_model = recommender.recommend(user_id, cutoff = cutoff)

            if training_dataframe.at[user_id, "ItemID"] is None:
                training_dataframe.at[user_id, "ItemID"] = recommendations_model
            else:
                training_dataframe.at[user_id, "ItemID"] = np.union1d(
                    training_dataframe.at[user_id, "ItemID"],
                    recommendations_model
                )

    training_dataframe = training_dataframe.explode("ItemID")
    return training_dataframe

In [ ]:
def get_user_batches(user_ids, batch_size=1000):
    for i in range(0, len(user_ids), batch_size):
        yield user_ids[i:i + batch_size]

def add_models_features(training_dataframe, URM_train):
    models = load_models(URM_train, models_mapping, model_folder="xg_boost_train")

    # Extract unique (UserID, ItemID) candidates
    feature_candidates = training_dataframe[['UserID', 'ItemID']].copy()
    N_CANDIDATES = len(feature_candidates)
    new_features_to_merge = [] 
    unique_users = feature_candidates['UserID'].unique()
    BATCH_SIZE = 1000 # Define your batch size here

    # --- 2. Refactored Feature Extraction Loop (Model by Model) ---
    for label, recommender in models:
        print(f"Processing features for model: {label}")
        
        # Pre-allocate arrays for the new features (only size of the candidate set)
        current_scores = np.zeros(N_CANDIDATES, dtype=np.float32)
        current_ranks = np.zeros(N_CANDIDATES, dtype=np.int32)
        
        # Iterate over user batches
        for user_batch in tqdm(list(get_user_batches(unique_users, BATCH_SIZE)), desc=f"Processing Batches for {label}", leave=False):
            
            scores_batch = recommender._compute_item_score(user_id_array=user_batch)

            # Normalize
            norm_factor = np.linalg.norm(scores_batch, np.inf, axis=1, keepdims=True) + 1e-6
            linf_scores_batch = scores_batch / norm_factor

            # Remove 
            for i, user_id in enumerate(user_batch):
                linf_scores_batch[i, :] = recommender._remove_seen_on_scores(user_id, linf_scores_batch[i, :])

            # Calculate rank
            rank_order = np.argsort(linf_scores_batch, axis=1)[:, ::-1]
            rank_matrix_batch = np.zeros_like(linf_scores_batch, dtype=np.int32)

            for i in range(len(user_batch)):
                rank_matrix_batch[i, rank_order[i, :]] = np.arange(len(scores_batch[i]))

            # Create a mapping from UserID to its index within the current batch (0 to N_batch-1)
            user_to_batch_index = {uid: i for i, uid in enumerate(user_batch)}
            
            # Get all candidates rows belonging to the current batch of users
            batch_mask = feature_candidates['UserID'].isin(user_batch)
            batch_candidate_indices = feature_candidates.index[batch_mask].to_numpy()
            
            # Identify ItemIDs and map UserIDs to the batch index
            candidate_item_ids = feature_candidates.loc[batch_candidate_indices, 'ItemID'].to_numpy().astype(np.int32)
            candidate_user_ids = feature_candidates.loc[batch_candidate_indices, 'UserID'].to_numpy()
            
            # Map the UserIDs to their row index in the N_batch dimension (0, 1, 2, ...)
            batch_row_indices = np.array([user_to_batch_index[uid] for uid in candidate_user_ids])
            
            # Perform the final indexed lookup (2D indexing on the batch matrices)
            current_scores[batch_candidate_indices] = linf_scores_batch[batch_row_indices, candidate_item_ids]
            current_ranks[batch_candidate_indices] = rank_matrix_batch[batch_row_indices, candidate_item_ids]
            
            # CRITICAL: Delete the temporary large matrices
            del scores_batch, linf_scores_batch, rank_matrix_batch, rank_order
            gc.collect()

        # Finalize and store the new features
        df_features = pd.DataFrame({
            f"{label}_Score": current_scores,
            f"{label}_RankPosition": current_ranks,
            f"{label}_Recommended": (current_ranks < 10).astype(int)
        })
        new_features_to_merge.append(df_features)
        
        # Cleanup for the next model
        del current_scores, current_ranks, df_features
        gc.collect()

    # Concatenate all new feature DataFrames along the columns axis
    all_new_features = pd.concat(new_features_to_merge, axis=1)

    # Add the new feature columns back to the main DataFrame
    for col in all_new_features.columns:
        training_dataframe[col] = all_new_features[col].values
        
    training_dataframe = training_dataframe.set_index('UserID') 

    # Final cleanup
    del all_new_features, feature_candidates
    gc.collect()


In [ ]:
def calculate_item_item_features_batched(training_dataframe, URM_train, batch_size=1000):
    similarity_models = load_models(
        URM_train,
        {
            'ItemKNNCF_cosine': ItemKNNCFRecommender,
            'ItemKNNCF_tversky': ItemKNNCFRecommender,
            'RP3beta': RP3betaRecommender
        },
        model_folder="xg_boost_train_similarity_features")
    
    stats_list = ['Avg', 'Max', 'Min', 'Std', 'Skew', 'Kurtosis']
    all_user_ids = training_dataframe.index.unique().to_numpy()
    
    # Store initial features to merge at the end
    df_features_to_merge = training_dataframe.reset_index()[['UserID', 'ItemID']].copy()
    N_CANDIDATES = len(df_features_to_merge)

    for similarity_type, recommender in similarity_models:
        print(f"Adding {similarity_type} similarity features (Batched & Sparse Optimized)...")
        
        # Use the SPARSE Similarity Matrix
        W_sparse = recommender.W_sparse.tocsr() 
        
        # Prepare storage for the new feature columns (size: N_candidates)
        new_features = {
            f"{stat}SimilarityToSeen{similarity_type}": np.zeros(N_CANDIDATES, dtype=np.float32) 
            for stat in stats_list
        }
        
        # Loop over user batches
        user_batches = list(get_user_batches(all_user_ids, batch_size))
        for user_batch in tqdm.tqdm(user_batches, desc=f"Processing Batches for {similarity_type}"):
            # Identify all candidates belonging to the current batch of users
            batch_mask = df_features_to_merge['UserID'].isin(user_batch)
            
            # Initialize storage for the stats arrays for this batch
            batch_stats_results = {stat: [] for stat in stats_list}
            
            for user_id in user_batch:
                # Get the candidate indices within the *full* df_features_to_merge for this *single* user
                user_candidate_indices = df_features_to_merge.index[
                    (df_features_to_merge['UserID'] == user_id) & batch_mask
                ].to_numpy()
                
                if len(user_candidate_indices) == 0:
                    continue # No candidates for this user in the training set
                
                # Items the user has interacted with
                seen_items = URM_train.getrow(user_id).nonzero()[1]
                
                # Get candidate item IDs for this user only
                current_candidate_item_ids = df_features_to_merge.loc[user_candidate_indices, 'ItemID'].values.astype(np.int32)
                
                if len(seen_items) == 0:
                    # Append zeros for all candidates of this user
                    num_candidates = len(user_candidate_indices)
                    for stat in stats_list:
                         batch_stats_results[stat].append(np.zeros(num_candidates, dtype=np.float32))
                    continue
                    
                # Extract Similarity Scores using SPARSE Slicing (Vectorized across N_candidates)
                similarities_candidate_rows = W_sparse[current_candidate_item_ids, :] 
                similarities_slice = similarities_candidate_rows[:, seen_items]
                similarities = similarities_slice.toarray()
                
                # Calculate statistics
                results = {
                    "Avg": similarities.mean(axis=1),
                    "Max": similarities.max(axis=1),
                    "Min": similarities.min(axis=1),
                    "Std": similarities.std(axis=1),
                    "Skew": stats.skew(similarities, axis=1),
                    "Kurtosis": stats.kurtosis(similarities, axis=1)
                }
                
                # Store results for later assignment
                for stat, value in results.items():
                    batch_stats_results[stat].append(value)

                del similarities, similarities_slice, similarities_candidate_rows
                gc.collect()

            #  Concatenate and assign the results for the whole batch
            if batch_stats_results['Avg']: # Check if any results were actually collected
                for stat in stats_list:
                    # Concatenate all user results for this stat into one array
                    stat_values = np.concatenate(batch_stats_results[stat])
                    
                    # Find the specific indices within the main feature array where these values go
                    batch_candidates_sorted = df_features_to_merge[batch_mask].index.to_numpy()
                    
                    # The gathered stat_values must correspond exactly to the order of batch_candidates_sorted
                    new_features[f"{stat}SimilarityToSeen{similarity_type}"][batch_candidates_sorted] = stat_values

        # Add the completed feature columns to the merge DataFrame
        for col_name, data in new_features.items():
            df_features_to_merge[col_name] = data

    # --- 5. Final Merge (Same as before) ---
    training_dataframe = pd.merge(training_dataframe.reset_index(), 
                                 df_features_to_merge, 
                                 on=['UserID', 'ItemID'], 
                                 how='left',
                                 suffixes=('_old', ''))

    training_dataframe = training_dataframe.set_index('UserID')
    
    del df_features_to_merge, W_sparse
    gc.collect()
    
    return training_dataframe

In [ ]:
def add_user_statistics(training_dataframe):
    # Consensus Features
    recommended_columns = [col for col in training_dataframe.columns if col.endswith('_Recommended')]
    training_dataframe['Counter_Recommended'] = training_dataframe[recommended_columns].sum(axis=1).astype(int)

    # Rank Position Statistics
    position_columns = [col for col in training_dataframe.columns if col.endswith('_RankPosition')]

    training_dataframe['Mean_RankPosition'] = training_dataframe[position_columns].mean(axis=1)
    training_dataframe['Std_RankPosition'] = training_dataframe[position_columns].std(axis=1)
    training_dataframe['Skew_RankPosition'] = training_dataframe[position_columns].skew(axis=1)
    training_dataframe['Kurtosis_RankPosition'] = training_dataframe[position_columns].kurtosis(axis=1)

    # Score Statistics
    score_columns = [col for col in training_dataframe.columns if col.endswith('_Score')]

    training_dataframe['Mean_Score'] = training_dataframe[score_columns].mean(axis=1)
    training_dataframe['Std_Score'] = training_dataframe[score_columns].std(axis=1)
    training_dataframe['Skew_Score'] = training_dataframe[score_columns].skew(axis=1)
    training_dataframe['Kurtosis_Score'] = training_dataframe[score_columns].kurtosis(axis=1)

    # Final Cleanup and Index Reset
    # We ensure the index is a regular column and any intermediate index is removed.
    # Assuming 'UserID' is the name of the index at this point from previous steps
    if training_dataframe.index.name == 'UserID':
        training_dataframe = training_dataframe.reset_index()
    else:
        # If index is unnamed (from a previous reset), just reset it and ensure columns are unique
        training_dataframe = training_dataframe.reset_index(drop=True)

    return training_dataframe

In [ ]:
def check_for_nans(df):
    assert 'UserID' in df.columns
    assert 'ItemID' in df.columns
    assert df.shape[0] == 6296650

    print("\n## ⚠️ Missing Value Check (Should be near zero)")
    print(df.isnull().sum().sort_values(ascending=False).head(10))

    print("\n## 🎯 Rank and Consensus Checks")
    # Max rank should be less than the total number of items (N_items)
    print(f"Max Rank Position: {df['ItemKNNCF_cosine_RankPosition'].max()}") 
    # Max consensus count should match the number of models used
    print(f"Max Recommended Counter: {df['Counter_Recommended'].max()}")

    print("\n## 📈 Similarity Feature Range Check")
    # Example for Cosine Avg Similarity
    print(f"Max Mean Cosine Sim: {df['AvgSimilarityToSeenItemKNN_cosine'].max()}")
    print(f"Min Mean Cosine Sim: {df['AvgSimilarityToSeenItemKNN_cosine'].min()}")

    print("\n## 📊 Meta-Feature Statistics (Descriptive)")
    print(df[['Mean_RankPosition', 'Std_RankPosition', 'Skew_RankPosition']].describe())
    print(df[['Mean_Score', 'Std_Score']].describe())

## **Training Dataframe**

### **Generate Candidates**

In [ ]:
training_dataframe = generate_candidates(URM_train)

In [ ]:
URM_validation_coo = sps.coo_matrix(URM_validation)

correct_recommendations = pd.DataFrame({"UserID": URM_validation_coo.row,
                                        "ItemID": URM_validation_coo.col})
correct_recommendations

In [ ]:
# Label the training dataframe
training_dataframe = pd.merge(training_dataframe, correct_recommendations, on=['UserID','ItemID'], how='left', indicator='Exist')
training_dataframe["Label"] = training_dataframe["Exist"] == "both"
training_dataframe.drop(columns = ['Exist'], inplace=True)
training_dataframe

In [ ]:
training_dataframe.reset_index()

In [ ]:
training_dataframe.Label.sum()

### **Add Models Features**

In [ ]:
training_dataframe = add_models_features(training_dataframe, URM_train)
training_dataframe

### **Similarity based features**

In [ ]:
training_dataframe = calculate_item_item_features_batched(training_dataframe, URM_train)

### **Sanity check**

In [ ]:
check_for_nans(training_dataframe)

In [ ]:
# Apply the fix immediately after the feature engineering steps
for col in training_dataframe.columns:
    if 'SimilarityToSeen' in col and training_dataframe[col].isnull().any():
        # Impute Skew/Kurtosis NaNs with 0.0
        training_dataframe[col] = training_dataframe[col].fillna(0.0)

print("NaNs imputed to 0.0.")

In [ ]:
check_for_nans(training_dataframe)

#### **User Statistics**

In [ ]:
training_dataframe = add_user_statistics(training_dataframe)

print("Feature Engineering Complete. 🎉")
print(f"Final DataFrame shape: {training_dataframe.shape}")

### **Save Dataframe**

In [ ]:
file_path = os.path.join(DATAFRAMES_PATH, "training.csv")
training_dataframe.to_csv(file_path, index=False)

## **Validation Dataframe**

### **Generate Candidates**

In [ ]:
training_dataframe = generate_candidates(URM_train_complete)

### **Add Models Features**

In [ ]:
training_dataframe = add_models_features(training_dataframe, URM_train_complete)
training_dataframe

### **Similarity based features**

In [ ]:
training_dataframe = calculate_item_item_features_batched(training_dataframe, URM_train_complete)

### **Sanity check**

In [ ]:
check_for_nans(training_dataframe)

In [ ]:
# Apply the fix immediately after the feature engineering steps
for col in training_dataframe.columns:
    if 'SimilarityToSeen' in col and training_dataframe[col].isnull().any():
        # Impute Skew/Kurtosis NaNs with 0.0
        training_dataframe[col] = training_dataframe[col].fillna(0.0)

print("NaNs imputed to 0.0.")

In [ ]:
check_for_nans(training_dataframe)

#### **User Statistics**

In [ ]:
training_dataframe = add_user_statistics(training_dataframe)

print("Feature Engineering Complete. 🎉")
print(f"Final DataFrame shape: {training_dataframe.shape}")

### **Save Dataframe**

In [ ]:
file_path = os.path.join(DATAFRAMES_PATH, "validation.csv")
training_dataframe.to_csv(file_path, index=False)

## **Prediction Dataframe**

In [ ]:
all_data = URM_train_complete + URM_test

### **Generate Candidates**

In [ ]:
training_dataframe = generate_candidates(all_data)

### **Add Models Features**

In [ ]:
training_dataframe = add_models_features(training_dataframe, all_data)
training_dataframe

### **Similarity based features**

In [ ]:
training_dataframe = calculate_item_item_features_batched(training_dataframe, all_data)

### **Sanity check**

In [ ]:
check_for_nans(training_dataframe)

In [ ]:
# Apply the fix immediately after the feature engineering steps
for col in training_dataframe.columns:
    if 'SimilarityToSeen' in col and training_dataframe[col].isnull().any():
        # Impute Skew/Kurtosis NaNs with 0.0
        training_dataframe[col] = training_dataframe[col].fillna(0.0)

print("NaNs imputed to 0.0.")

In [ ]:
check_for_nans(training_dataframe)

#### **User Statistics**

In [ ]:
training_dataframe = add_user_statistics(training_dataframe)

print("Feature Engineering Complete. 🎉")
print(f"Final DataFrame shape: {training_dataframe.shape}")

### **Save dataframe**

In [ ]:
file_path = os.path.join(DATAFRAMES_PATH, "prediction.csv")
training_dataframe.to_csv(file_path, index=False)